# Exploring HEST-1k Xenium data (TENX96, TENX99)

In [ ]:
import os

import numpy as np
import pandas as pd
import geopandas as gpd
import h5py
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from shapely.geometry import box
from IPython.display import display

from hest import iter_hest

HEST_DIR = "/lustre/scratch126/cellgen/lotfollahi/so25/data/hest_samples"
SAMPLE_IDS = ["TENX96", "TENX99"]

plt.rcParams["figure.dpi"] = 110

In [ ]:
# HEST-Library's own loader that returns a HESTData object per sample.
samples = {st.meta["id"]: st for st in iter_hest(HEST_DIR, id_list=SAMPLE_IDS)}
list(samples.keys())

In [ ]:
samples.items()

In [ ]:
def panel_label(meta):
    predesigned = meta.get("panel_num_targets_predesigned", 0)
    custom = meta.get("panel_num_targets_custom", 0)
    return f"Xenium ({predesigned} predesigned + {custom} custom)"


rows = []
for sid, st in samples.items():
    m = st.meta
    rows.append(
        {
            "sample_id": sid,
            "organ": m["organ"],
            "disease": f'{m["disease_state"]} ({m["oncotree_code"]})',
            "panel": panel_label(m),
            "panel_name": m["panel_name"],
            "n_genes_panel": st.adata.n_vars,
            "n_cells_ST_matrix": st.adata.n_obs,
            "n_cells_xenium_segmentation": m["cells_under_tissue"],
            "wsi_width_px": m["fullres_px_width"],
            "wsi_height_px": m["fullres_px_height"],
            "pixel_size_um_per_px": m["pixel_size_um_estimated"],
            "scan_magnification": m["magnification"],
        }
    )

inventory = pd.DataFrame(rows).set_index("sample_id")
inventory

In [ ]:
for sid, st in samples.items():
    print(f"=== {sid} ===")
    print(st.adata)
    print()

HEST-1k pools Xenium single-cell data onto a **~100 micron pseudo-spot grid**,
matching Visium's spot pitch, so that Xenium and Visium samples can be
benchmarked on a common footing. Each row in `st.adata.obs` is therefore a
pooled bin containing on the order of dozens of real cells, not a single cell.
We confirm the ~100 um grid spacing directly below.

In [ ]:
# Confirm the ST object sits on a ~100 um grid rather than single-cell resolution
st96 = samples["TENX96"]
xs = np.sort(st96.adata.obs["pxl_col_in_fullres"].unique())
spacing_um = np.median(np.diff(xs)) * st96.pixel_size
print(
    f"Median pseudo-spot spacing: {spacing_um:.1f} um (Visium-like grid, not single-cell)"
)

for sid, st in samples.items():
    n_pooled = st.adata.n_obs
    n_true_cells = st.meta["cells_under_tissue"]
    print(
        f"{sid}: {n_pooled} pooled ST spots vs {n_true_cells} true Xenium cells "
        f"({n_true_cells / n_pooled:.1f}x more true cells than pooled spots)"
    )

In [ ]:
# Total transcript counts and genes detected, per pooled ST spot
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for col, (sid, st) in enumerate(samples.items()):
    axes[0, col].hist(st.adata.obs["total_counts"], bins=50, color="steelblue")
    axes[0, col].set_title(f"{sid}: transcript counts / pseudo-spot")
    axes[0, col].set_xlabel("total_counts")

    axes[1, col].hist(st.adata.obs["n_genes_by_counts"], bins=50, color="indianred")
    axes[1, col].set_title(f"{sid}: genes detected / pseudo-spot")
    axes[1, col].set_xlabel("n_genes_by_counts")
fig.tight_layout()

Here `n_genes_by_counts` means how many spots there are (y-axis) with X many distinct genes detected (x-axis) and not their count.

In [ ]:
# Top 20 most-expressed genes per sample
for sid, st in samples.items():
    print(f"=== {sid}: top 20 genes by total counts ===")
    top20 = st.adata.var.sort_values("total_counts", ascending=False).head(20)
    display(top20[["total_counts", "mean_counts", "n_cells_by_counts"]])

### Gene panel differs between the two samples

Both samples report `n_genes_panel = 541`, but that count includes Xenium's
negative-control probes (`NegControlProbe_*`, `NegControlCodeword_*`, `BLANK_*`)
alongside real genes — they are not all genes. We separate the two below.

In [ ]:
def is_control_probe(name):
    return name.startswith(("NegControlProbe_", "NegControlCodeword_", "BLANK_"))


for sid, st in samples.items():
    n_ctrl = sum(is_control_probe(g) for g in st.adata.var_names)
    print(
        f"{sid}: {st.adata.n_vars} panel entries, {n_ctrl} control probes, "
        f"{st.adata.n_vars - n_ctrl} real genes"
    )

g96 = {g for g in samples["TENX96"].adata.var_names if not is_control_probe(g)}
g99 = {g for g in samples["TENX99"].adata.var_names if not is_control_probe(g)}
print()
print(f"Shared real genes: {len(g96 & g99)}")
print(f"Real genes unique to TENX96: {len(g96 - g99)}")
print(f"Real genes unique to TENX99: {len(g99 - g96)}")

### Transcript-level QC: detection confidence (`qv`) and control-probe call rate

Beyond panel identity, the raw per-transcript file (`transcripts/*.parquet`)
carries a `qv` column — a Phred-scaled quality score for each individual
transcript call — plus `cell_id` (`-1` if the call wasn't assigned to any
cell) and `he_x`/`he_y` (its position on the H&E, already non-rigidly
registered by HEST — no extra alignment step needed to plot it). We use `qv`
to get an empirical noise estimate: negative-control probes should almost
never represent real biology, so their pass rate at Xenium's standard
`qv >= 20` cutoff (10x's own default for building the cell x gene matrix) is
a direct measure of the assay's false-positive rate.

In [ ]:
# Read only the columns we need from the 53M-127M row transcript files
QV_THRESHOLD = 20  # Xenium's standard confidence cutoff (10x default)

transcripts = {}
for sid in SAMPLE_IDS:
    df = pd.read_parquet(
        os.path.join(HEST_DIR, "transcripts", f"{sid}_transcripts.parquet"),
        columns=["cell_id", "feature_name", "he_x", "he_y", "qv"],
    )
    transcripts[sid] = df
    print(f"{sid}: {len(df):,} raw transcript calls")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, sid in zip(axes, SAMPLE_IDS):
    ax.hist(transcripts[sid]["qv"], bins=50, color="slategray")
    ax.axvline(
        QV_THRESHOLD, color="red", linestyle="--", label=f"qv={QV_THRESHOLD} cutoff"
    )
    ax.set_title(f"{sid}: transcript quality score (qv)")
    ax.set_xlabel("qv")
    ax.legend()
fig.tight_layout()

In [ ]:
# Empirical false-positive rate: control probes passing the qv cutoff are
# necessarily noise, since they have no real biological target
for sid in SAMPLE_IDS:
    df = transcripts[sid]
    is_ctrl = df["feature_name"].astype(str).map(is_control_probe)
    for label, mask in [("real genes", ~is_ctrl), ("control probes", is_ctrl)]:
        sub = df.loc[mask, "qv"]
        pass_rate = (sub >= QV_THRESHOLD).mean()
        print(
            f"{sid} {label}: {len(sub):,} calls, {100 * pass_rate:.1f}% pass qv>={QV_THRESHOLD}"
        )
    frac_unassigned = (df["cell_id"] == -1).mean()
    print(
        f"{sid}: {100 * frac_unassigned:.1f}% of transcripts unassigned to any cell (cell_id == -1)"
    )
    print()

### H&E side

The full-resolution WSI is exposed as `st.wsi` (a `trident` WSI object wrapped
by HEST), lazily backed by the pyramidal `.tif`. We report the pyramid levels,
show a thumbnail, and display a handful of the 224x224 px / 0.5 um/px (20x)
patches HEST pre-extracted into `patches/*.h5`.

Note: the metadata's `magnification: 40x` refers to the *scan* magnification
(native pixel size 0.2125 um/px). The 224x224 patches requested at "20x"
(0.5 um/px) are a 2x downsample of that native resolution — HEST already
extracted them this way (`patch_size_src=527` native px -> `224` px target,
`527 * 0.2125 um = 224 * 0.5 um = 112 um`, confirming the patches are exactly
20x-equivalent).

In [ ]:
rows = []
for sid, st in samples.items():
    wsi = st.wsi
    for lvl, (dim, ds) in enumerate(zip(wsi.level_dimensions, wsi.level_downsamples)):
        rows.append(
            {
                "sample_id": sid,
                "level": lvl,
                "width": dim[0],
                "height": dim[1],
                "downsample": round(ds, 2),
            }
        )
pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (sid, st) in zip(axes, samples.items()):
    thumb = st.wsi.get_thumbnail((600, 600))
    ax.imshow(thumb)
    ax.set_title(f"{sid}: thumbnail")
    ax.axis("off")
fig.tight_layout()

In [ ]:
# 4 random pre-extracted 224x224 px patches (0.5 um/px, i.e. 20x) per sample
rng = np.random.default_rng(0)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, sid in enumerate(SAMPLE_IDS):
    with h5py.File(os.path.join(HEST_DIR, "patches", f"{sid}.h5"), "r") as f:
        idx = np.sort(rng.choice(f["img"].shape[0], size=4, replace=False))
        for col, i in enumerate(idx):
            axes[row, col].imshow(f["img"][i])
            axes[row, col].set_title(f"{sid} patch #{i}")
            axes[row, col].axis("off")
fig.suptitle("Random 224x224 px patches at 20x (0.5 um/px)")
fig.tight_layout()

### Alignment check

We overlay every pooled ST spot centroid on the downscaled thumbnail. Then we zoom into a genuinely dense ~500x500 um region at full resolution and overlay individual Xenium cell centroids and CellViT nuclear boundaries directly on the H&E pixels.

In [ ]:
# Load the true single-cell segmentation (already in full-res H&E pixel space)
xenium_cells = {}
cellvit_cells = {}
for sid in SAMPLE_IDS:
    xc = gpd.read_parquet(
        os.path.join(HEST_DIR, "xenium_seg", f"{sid}_xenium_cell_seg.parquet")
    )
    xc["cx"] = xc.geometry.centroid.x
    xc["cy"] = xc.geometry.centroid.y
    xenium_cells[sid] = xc

    cv = gpd.read_parquet(
        os.path.join(HEST_DIR, "cellvit_seg", f"{sid}_cellvit_seg.parquet")
    )
    cv["cx"] = cv.geometry.centroid.x
    cv["cy"] = cv.geometry.centroid.y
    cellvit_cells[sid] = cv

{sid: (len(xenium_cells[sid]), len(cellvit_cells[sid])) for sid in SAMPLE_IDS}

In [ ]:
# Step 1: overlay ST pseudo-spot centroids on the downscaled thumbnail
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (sid, st) in zip(axes, samples.items()):
    scalef = st.adata.uns["spatial"]["ST"]["scalefactors"][
        "tissue_downscaled_fullres_scalef"
    ]
    thumb = st.adata.uns["spatial"]["ST"]["images"]["downscaled_fullres"]
    xy = st.adata.obsm["spatial"] * scalef

    ax.imshow(thumb)
    ax.scatter(xy[:, 0], xy[:, 1], s=3, c="red", alpha=0.6)
    ax.set_title(f"{sid}: ST spot centroids on H&E thumbnail")
    ax.axis("off")
fig.tight_layout()

In [ ]:
# Find a genuinely dense region to zoom into, using true single-cell centroids
def find_dense_region_center(cells_gdf, bin_size_um=50, pixel_size=0.2125):
    bin_px = bin_size_um / pixel_size
    xb = (cells_gdf["cx"] // bin_px).astype(int)
    yb = (cells_gdf["cy"] // bin_px).astype(int)
    top_bin = pd.DataFrame({"xb": xb, "yb": yb}).value_counts().index[0]
    xb0, yb0 = top_bin
    return (xb0 + 0.5) * bin_px, (yb0 + 0.5) * bin_px


dense_centers = {
    sid: find_dense_region_center(xenium_cells[sid], pixel_size=samples[sid].pixel_size)
    for sid in SAMPLE_IDS
}
dense_centers

In [ ]:
def get_crop(st, center_xy, size_um):
    px = st.pixel_size
    size_px = int(round(size_um / px))
    cx, cy = center_xy
    x0 = int(cx - size_px / 2)
    y0 = int(cy - size_px / 2)
    region = st.wsi.read_region(
        (x0, y0), level=0, size=(size_px, size_px), read_as="numpy"
    )
    return region, x0, y0, size_px

In [ ]:
# Step 2: zoom to a 500x500 um crop at full resolution, overlay true cell
# centroids and CellViT nuclear boundaries directly on the H&E pixels
fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for col, sid in enumerate(SAMPLE_IDS):
    st = samples[sid]
    region, x0, y0, size_px = get_crop(st, dense_centers[sid], size_um=500)

    axes[0, col].imshow(region)
    axes[0, col].set_title(f"{sid}: H&E only (500x500 um)")
    axes[0, col].axis("off")

    axes[1, col].imshow(region)

    xc = xenium_cells[sid]
    in_view = xc["cx"].between(x0, x0 + size_px) & xc["cy"].between(y0, y0 + size_px)
    axes[1, col].scatter(
        xc.loc[in_view, "cx"] - x0,
        xc.loc[in_view, "cy"] - y0,
        s=8,
        c="lime",
        label="Xenium cell centroid",
    )

    cv = cellvit_cells[sid]
    cv_in_view = cv["cx"].between(x0, x0 + size_px) & cv["cy"].between(y0, y0 + size_px)
    for geom in cv.loc[cv_in_view, "geometry"]:
        ex, ey = geom.exterior.xy
        axes[1, col].plot(np.array(ex) - x0, np.array(ey) - y0, c="red", linewidth=0.6)

    axes[1, col].set_title(f"{sid}: + Xenium centroids (green) & CellViT nuclei (red)")
    axes[1, col].axis("off")
    axes[1, col].legend(loc="lower right", fontsize=8)
    axes[1, col].add_artist(ScaleBar(st.pixel_size, "um", location="lower left"))
fig.tight_layout()

### Individual transcript detections on the H&E crop

Cell centroids are already a good alignment check, but individual transcript calls (`he_x`/`he_y`) are a finer one: real detections should cluster tightly on visible cell bodies.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
for ax, sid in zip(axes, SAMPLE_IDS):
    st = samples[sid]
    region, x0, y0, size_px = get_crop(st, dense_centers[sid], size_um=100)
    ax.imshow(region)

    df = transcripts[sid]
    in_view = (
        df["he_x"].between(x0, x0 + size_px)
        & df["he_y"].between(y0, y0 + size_px)
        & (df["qv"] >= QV_THRESHOLD)
    )
    sub = df.loc[in_view]
    is_ctrl = sub["feature_name"].astype(str).map(is_control_probe)

    ax.scatter(
        sub.loc[~is_ctrl, "he_x"] - x0,
        sub.loc[~is_ctrl, "he_y"] - y0,
        s=6,
        c="orange",
        alpha=0.7,
        label="real gene transcript",
    )
    ax.scatter(
        sub.loc[is_ctrl, "he_x"] - x0,
        sub.loc[is_ctrl, "he_y"] - y0,
        s=25,
        c="yellow",
        marker="x",
        label="control-probe transcript",
    )

    ax.set_title(f"{sid}: individual transcript calls (qv>={QV_THRESHOLD})")
    ax.axis("off")
    ax.legend(loc="lower right", fontsize=7)
    ax.add_artist(ScaleBar(st.pixel_size, "um", location="lower left"))
fig.tight_layout()

### Cell segmentation

Two independent segmentations ship with each sample:
- **CellViT** — a deep-learning nuclear segmenter run directly on the H&E image.
- **Xenium native** — DAPI-based cell/nucleus calls from the Xenium instrument,
  non-rigidly aligned to H&E by HEST.

In [ ]:
# CellViT nuclear boundaries on a small (100x100 um) H&E crop, colored by class
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
for ax, sid in zip(axes, SAMPLE_IDS):
    st = samples[sid]
    region, x0, y0, size_px = get_crop(st, dense_centers[sid], size_um=100)
    ax.imshow(region)

    cv = cellvit_cells[sid]
    in_view = cv["cx"].between(x0, x0 + size_px) & cv["cy"].between(y0, y0 + size_px)
    for _, row in cv.loc[in_view].iterrows():
        ex, ey = row.geometry.exterior.xy
        color = "yellow" if row["class"] == "Neoplastic" else "cyan"
        ax.plot(np.array(ex) - x0, np.array(ey) - y0, c=color, linewidth=1)

    ax.set_title(f"{sid}: CellViT nuclei (yellow=Neoplastic, cyan=other)")
    ax.axis("off")
    ax.add_artist(ScaleBar(st.pixel_size, "um", location="lower left"))
fig.tight_layout()

In [ ]:
# Three different "cell count" numbers, and why they differ
rows = []
for sid, st in samples.items():
    n_cellvit = len(cellvit_cells[sid])
    n_xenium = len(xenium_cells[sid])
    n_st = st.adata.n_obs
    rows.append(
        {
            "sample_id": sid,
            "cellvit_nuclei_on_HE": n_cellvit,
            "xenium_native_cells": n_xenium,
            "ST_matrix_pseudo_spots": n_st,
            "cellvit_vs_xenium_native_pct_diff": round(
                100 * (n_cellvit - n_xenium) / n_xenium, 1
            ),
        }
    )
pd.DataFrame(rows).set_index("sample_id")

In [ ]:
# CellViT cell-type composition, for context
for sid in SAMPLE_IDS:
    print(f"=== {sid}: CellViT class composition ===")
    print(cellvit_cells[sid]["class"].value_counts())
    print()

`cellvit_nuclei_on_HE` and `xenium_native_cells` are two independent estimates
of the *same* underlying single-cell count — one from an image-only nuclear
detector, one from the Xenium instrument's own DAPI-based caller aligned onto
the H&E.

### Spatial structure

We plot at the same pooled ST spot resolution used in Section 2 because gene expression values only exist at that resolution in the `AnnData`.

Markers: **EPCAM** (epithelial/tumor), **PTPRC** (immune, CD45), **PECAM1**
(endothelial, CD31).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (sid, st) in zip(axes, samples.items()):
    xy = st.adata.obsm["spatial"]
    sca = ax.scatter(
        xy[:, 0], xy[:, 1], c=st.adata.obs["total_counts"], s=5, cmap="viridis"
    )
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_title(f"{sid}: total transcript counts per pseudo-spot")
    ax.axis("off")
    fig.colorbar(sca, ax=ax, shrink=0.7)
fig.tight_layout()

In [ ]:
markers = ["EPCAM", "PTPRC", "PECAM1"]
fig, axes = plt.subplots(
    len(SAMPLE_IDS), len(markers), figsize=(4 * len(markers), 4 * len(SAMPLE_IDS))
)
for row, (sid, st) in enumerate(samples.items()):
    xy = st.adata.obsm["spatial"]
    for col, gene in enumerate(markers):
        expr = np.asarray(st.adata[:, gene].X).flatten()
        ax = axes[row, col]
        sca = ax.scatter(xy[:, 0], xy[:, 1], c=expr, s=5, cmap="magma")
        ax.invert_yaxis()
        ax.set_aspect("equal")
        ax.set_title(f"{sid}: {gene}")
        ax.axis("off")
fig.tight_layout()

### Patch-level cell density

We tile each WSI into a fresh, non-overlapping grid of 224x224 px patches at
20x (0.5 um/px — i.e. 527x527 px in the native 0.2125 um/px image), keep only
patches that overlap the tissue mask (`st.tissue_contours`), and count true
single-cell centroids (`xenium_seg`) landing in each patch.


In [ ]:
def tile_wsi(st, tissue_gdf, target_patch_px=224, target_pixel_size_um=0.5):
    native_stride = int(round(target_patch_px * target_pixel_size_um / st.pixel_size))
    width, height = st.wsi.dimensions
    xs = np.arange(0, width, native_stride)
    ys = np.arange(0, height, native_stride)
    boxes = [box(x, y, x + native_stride, y + native_stride) for y in ys for x in xs]
    patches = gpd.GeoDataFrame({"geometry": boxes})

    tissue_union = tissue_gdf.union_all()
    patches = patches[patches.intersects(tissue_union)].reset_index(drop=True)
    return patches, native_stride

In [ ]:
density_results = {}
for sid, st in samples.items():
    patches, stride = tile_wsi(st, st.tissue_contours)

    centroids = xenium_cells[sid].copy()
    centroids["geometry"] = gpd.points_from_xy(centroids["cx"], centroids["cy"])

    joined = gpd.sjoin(centroids, patches, predicate="within", how="inner")
    counts = joined.groupby("index_right").size().reindex(patches.index, fill_value=0)
    density_results[sid] = counts

    print(
        f"{sid}: {len(patches)} tissue-overlapping patches (stride={stride} native px), "
        f"median={counts.median():.0f}, p95={counts.quantile(0.95):.0f}, max={counts.max()}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, sid in zip(axes, SAMPLE_IDS):
    counts = density_results[sid]
    ax.hist(counts, bins=50, color="teal")
    ax.axvline(counts.median(), color="black", linestyle="--", label="median")
    ax.axvline(counts.quantile(0.95), color="red", linestyle="--", label="95th pct")
    ax.set_title(f"{sid}: cells per 224x224 px (20x) patch")
    ax.set_xlabel("cell count")
    ax.legend()
fig.tight_layout()

In [ ]:
pd.DataFrame(
    {
        sid: {
            "n_tissue_patches": len(density_results[sid]),
            "median_cells_per_patch": density_results[sid].median(),
            "p95_cells_per_patch": density_results[sid].quantile(0.95),
            "max_cells_per_patch": density_results[sid].max(),
        }
        for sid in SAMPLE_IDS
    }
).T